# Three Ways to Build RAG — and Why Only One of Them *Remembers*

> **ChromaDB** vs **PageIndex** vs **Karpathy's Self-Writing Wiki** — head to head on the same PDF, scored on 10 questions.

We're going to build three RAG systems and race them.

1. **Classic Vector RAG** — Docling parses the PDF, we chunk it, embed it, shove it in ChromaDB, retrieve top-k, stuff into context. The thing everyone ships.
2. **Vectorless RAG (PageIndex)** — No embeddings. No vector DB. PageIndex's hosted service builds a hierarchical tree of the document (title + summary + page range per node) and Claude *reasons* over it to the answer. No `similarity()` call anywhere.
3. **Self-Writing Wiki** (Karpathy pattern) — The LLM reads the PDF *once* and writes a markdown wiki. Queries read the wiki, not the PDF. Knowledge compounds across sources.

All three systems query the **same PDF**: Rahul Pandey's **"What does it take to be a data-driven organization?"** (DSciEr / Medium, 2023). A 12-minute strategy overview covering data-first culture, lakehouse architecture, data + AI governance, data-as-product, data mesh, the AI Center of Excellence, and sustainable AI — plenty of hierarchy, plenty of cross-references, plenty of named lists. Exactly the kind of structured prose that makes the differences between vector RAG, PageIndex, and a wiki visible.

Then we ask all three the same 10 questions. Winner takes the aux.

**No Noise. Just Build.** 🔨


---

## Section 0 — Setup

One API key. One PDF. Three RAG systems. Let's go.

### 0.1 Install dependencies

This project uses **uv**. Everything is pinned in `pyproject.toml`:
`anthropic`, `chromadb`, `sentence-transformers`, `docling`, `pandas`, `matplotlib`, `python-dotenv`, `jupyter`, `ipykernel`.

Run `uv sync` once from the project root, then register the kernel (command below), then come back to this notebook.


In [ ]:
# Dependencies are managed by `uv`. From the project root:
#     uv sync
# That reads pyproject.toml and installs everything into .venv/.
#
# Then attach the kernel to this project's venv:
#     uv run python -m ipykernel install --user --name rag2-0 --display-name "rag2-0 (uv)"
# and pick "rag2-0 (uv)" as the notebook kernel.
#
# First run is slow: docling lazy-downloads a ~300 MB structural model on first PDF parse,
# and sentence-transformers grabs ~90 MB for all-MiniLM-L6-v2.


### 0.2 Config

The only external thing you need: an `ANTHROPIC_API_KEY` (drop it in `.env` next to this notebook — `load_dotenv()` will pick it up). The whole notebook runs on Claude — no OpenAI, no Pinecone, no hosted PageIndex API.

Three model tiers, three jobs:

- **Haiku** — fast + cheap. We use it for grunt work: summarizing each section in §2, picking which tree node to read, choosing wiki pages at query time. Anything where the *shape* of the output matters more than deep reasoning.
- **Sonnet** — the middleweight. Handles all the final answers and the big wiki-authoring pass in §3. Good-enough reasoning, reasonable cost.
- **Opus** — the heavy. We only spend it in §4 as the judge, because comparing three answers fairly is harder than generating any one of them.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# ---- API keys ----
# `.env` lives next to this notebook. load_dotenv() is a no-op if the file is absent,
# so an explicit shell `export ANTHROPIC_API_KEY=...` still works.
load_dotenv()
assert os.environ.get('ANTHROPIC_API_KEY'), (
    'ANTHROPIC_API_KEY is missing. '
    'Copy .env.example to .env and paste your key, or export it in the shell.'
)
if not os.environ.get('PAGEINDEX_API_KEY'):
    print('NOTE: PAGEINDEX_API_KEY not set — §2 (hosted PageIndex tree-build) will fail until you add it.')
    print('      Free key at https://dash.pageindex.ai/api-keys, then add to .env: PAGEINDEX_API_KEY=...')

# ---- Models ----
MODEL_CHEAP  = "claude-haiku-4-5-20251001"   # retrieval / tree-walk / wiki-ingest sub-calls
MODEL_ANSWER = "claude-sonnet-4-6"           # final answer generation
MODEL_JUDGE  = "claude-opus-4-7"             # scoring in Section 4 (strongest judge)

# ---- Paths ----
HERE       = Path('.').resolve()
PDF_PATH   = HERE / 'article.pdf'
VAULT      = HERE / 'Knowledge-Base'  # committed vault skeleton per CLAUDE.md §2
assert (VAULT / 'CLAUDE.md').exists(), (
    'Knowledge-Base/CLAUDE.md is missing. The vault skeleton should be committed.'
)

assert PDF_PATH.exists(), f"Missing {PDF_PATH}. Drop article.pdf next to this notebook."
print(f"PDF:   {PDF_PATH.name}  ({PDF_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"Vault: {VAULT.relative_to(HERE)}/")


### 0.3 The 10 questions (defined once, used everywhere)

Each question is tagged by what it *stresses* — the type of retrieval failure it's designed to expose. This isn't an accuracy benchmark; it's a **diagnostic**. Random questions tend to expose a system's average behavior. Targeted questions expose *where each architecture breaks*, which is the interesting thing if you care about picking one for a real project.

Read the `stresses` field on each entry — that's my prediction of who should win or lose on that question. §4.5 checks whether the predictions held up.


In [ ]:
QUESTIONS = [
    {
        "id": "Q1", "type": "factual-lookup",
        "question": "Who wrote this article, and what is the author's professional role?",
        "expected_keywords": ["rahul pandey", "mlops", "adidas"],
        "stresses": "Direct factual lookup. Author info appears in the byline and the bio block. All systems should handle.",
    },
    {
        "id": "Q2", "type": "enumeration",
        "question": "What are the four fundamental principles of data mesh according to the article?",
        "expected_keywords": ["domain-oriented", "decentralized", "data as a product", "self-serve", "federated"],
        "stresses": "Multi-point list from a single labeled section. Wiki should ace if its ingest captured the bullet list.",
    },
    {
        "id": "Q3", "type": "enumeration",
        "question": "What are the Key Areas the article says an AI Center of Excellence (CoE) should focus on?",
        "expected_keywords": ["strategy", "governance", "tools", "talent", "adoption"],
        "stresses": "Multi-item list synthesis (5 bullets). Wiki should ace.",
    },
    {
        "id": "Q4", "type": "cross-reference",
        "question": "The lakehouse architecture concept is attributed to a research paper in the References section. Who are the listed authors and which institutions are they affiliated with?",
        "expected_keywords": ["armbrust", "ghodsi", "xin", "zaharia", "databricks", "berkeley", "stanford"],
        "stresses": "Cross-reference between body text (which mentions \"lakehouse\") and the References section (which has the citation). PageIndex-style tree navigation should shine; vector RAG often misses Reference citations because they're semantically far from the topical chunks.",
    },
    {
        "id": "Q5", "type": "procedural",
        "question": "What concrete actions does the article recommend to foster a data-first culture?",
        "expected_keywords": ["leadership", "literacy", "training", "collaboration", "experimentation", "celebrate"],
        "stresses": "Procedural list (8 bullets), spread across one section. Tests whether retrieval surfaces the whole list, not just one or two adjacent bullets.",
    },
    {
        "id": "Q6", "type": "multi-step-synthesis",
        "question": "How are the lakehouse architecture and data mesh related, according to the article?",
        "expected_keywords": ["lakehouse", "mesh", "unified", "decentralized", "governance"],
        "stresses": "Synthesis spanning the Platform architecture section and the Treating-data-as-a-product section. Wiki should win because it pre-synthesized this relationship at ingest time.",
    },
    {
        "id": "Q7", "type": "quantitative",
        "question": "The article cites two specific predictions — one from PwC about AI's economic impact, and one from Gartner about data governance obstacles. What are the numbers and the target years?",
        "expected_keywords": ["15.7", "trillion", "2030", "80%", "2025"],
        "stresses": "Two distinct quantitative facts in different sections (intro vs governance). Vector RAG may surface only one of the two because they live in semantically separated chunks.",
    },
    {
        "id": "Q8", "type": "causal",
        "question": "Why does the article argue that centralized data platform teams have limitations?",
        "expected_keywords": ["resource", "constraint", "delivery", "infrastructure", "governance"],
        "stresses": "Causal chain (constraint → delays → shadow infra → governance breakdown) concentrated in one subsection. PageIndex should ace by jumping straight to \"Decouple platform and data ownership\".",
    },
    {
        "id": "Q9", "type": "out-of-scope",
        "question": "What does the article recommend for using generative AI to design quantum computing algorithms?",
        "expected_keywords": ["not", "does not", "no"],
        "stresses": "Negative / out-of-scope. Generative AI is mentioned (in the governance section) but quantum computing is never discussed. Vector RAG may hallucinate by pulling generic AI chunks.",
    },
    {
        "id": "Q10", "type": "meta-thesis",
        "question": "What is the article's overall thesis about what it takes for an organization to become truly data-driven?",
        "expected_keywords": ["more than", "culture", "architecture", "governance", "product", "mesh"],
        "stresses": "Big-picture synthesis spanning all sections. Wiki should win because the ingest pre-synthesizes the overview; vector RAG can only paraphrase a single chunk.",
    },
]
print(f"Loaded {len(QUESTIONS)} questions across {len({q['type'] for q in QUESTIONS})} stress types.")


### 0.4 Utility — call Claude once, cleanly

Thin wrapper so every section uses the same interface and we can count tokens and time. Those two numbers are what §4's scorecard uses to compare cost and latency — so all three systems call through here to keep the comparison apples-to-apples.


In [ ]:
import anthropic, time, re, json

client = anthropic.Anthropic()

def ask_claude(prompt, *, model=MODEL_ANSWER, system=None, max_tokens=2048,
               stream=False, cache_system=False):
    """One Claude call. Returns (anthropic.Message, latency_s).

    Callers use the SDK object directly: `msg.content[0].text`, `msg.usage.input_tokens`, etc.

    stream=True       — print a dot per ~500 chars while tokens arrive (UX for the long ingest call).
    cache_system=True — mark the system prompt with cache_control: ephemeral (90% discount + faster TTFT on reruns within 5 min).
    """
    sys_arg = ([{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}]
               if system and cache_system else system)
    args = dict(model=model, max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}],
                **({"system": sys_arg} if sys_arg else {}))
    t0 = time.time()
    if stream:
        chars, next_dot = 0, 500
        with client.messages.stream(**args) as s:
            for chunk in s.text_stream:
                chars += len(chunk)
                while chars >= next_dot:
                    print('.', end='', flush=True); next_dot += 500
            msg = s.get_final_message()
        print(f' [{chars:,} chars]')
    else:
        msg = client.messages.create(**args)
    return msg, time.time() - t0


def parse_json_blob(text, *, fallback=None, array=False):
    """Extract the first JSON object (or array) embedded in `text`. Tolerant of stray prose / fences.

    Used by every cell that asks Claude to return JSON. Returns `fallback` (default {}) on parse failure.
    """
    if fallback is None:
        fallback = [] if array else {}
    pattern = r'\[[\s\S]*\]' if array else r'\{[\s\S]*\}'
    m = re.search(pattern, text)
    if not m:
        return fallback
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return fallback


# quick smoke test (comment out if you don't want to spend a token)
# msg, dt = ask_claude("Say hi in 5 words.", model=MODEL_CHEAP, max_tokens=50)
# print(msg.content[0].text, '|', msg.usage.input_tokens, 'in /', msg.usage.output_tokens, 'out |', f'{dt:.2f}s')


---

## Section 1 — Classic Vector RAG (ChromaDB + Docling)

The thing every tutorial teaches. Five steps:

```
PDF  →  Docling  →  Markdown  →  chunks  →  embeddings  →  ChromaDB
                                                              ↓
                                        query → top-k → context → Claude → answer
```

The bet: *semantic similarity between query and chunk ≈ relevance*. When that bet holds, this works. When it doesn't, this is the system you're racing *against*.

### 1.1 PDF → Markdown via Docling (this is the vault's "ingest to Raw/" step)

Docling parses PDFs structurally (preserves headings, tables, lists) instead of dumping raw text. Worth it for everything downstream.

We write the output to `Knowledge-Base/Raw/article.md` — which *is* the first step from `CLAUDE.md` §0 ("new source lands in `Raw/`"). §1, §2, and §3 all read from there, which is how we keep the comparison apples-to-apples: same bytes go into all three systems.


In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
doc = converter.convert(str(PDF_PATH)).document
markdown_text = doc.export_to_markdown()

# This is the canonical landing spot for the parsed source. Raw/ is read-only to Claude
# (per CLAUDE.md §10); we only write here from this one cell, on purpose.
md_path = VAULT / 'Raw' / 'article.md'
md_path.write_text(markdown_text, encoding='utf-8')

print(f"Markdown: {len(markdown_text):,} chars across ~{markdown_text.count(chr(10)) + 1} lines")
print(f"Written:  {md_path.relative_to(HERE)}")
print("--- first 500 chars ---")
print(markdown_text[:500])


### 1.2 Chunk

Simple recursive splitter: walk paragraphs, accumulate until you hit the char budget (1200), start a new chunk with the last 150 chars of the previous one as a seed. The overlap is what keeps a sentence that happens to straddle a boundary from getting split mid-idea.

Two things to internalize here — they explain most of this system's failures:

1. **Chunks lose their neighbors.** The retriever gets chunk #42 but not #41 or #43. If the answer requires joining information across a boundary, vector RAG has to hope both chunks get retrieved (often they don't — they look semantically similar to each other, so top-k picks one and skips the other as redundant).
2. **Chunks lose their parent context.** A chunk containing "This approach works at scale" has no idea *which* approach or *what* scale — that was in the heading three paragraphs up. Docling's markdown keeps headings inline, which helps, but not fully.

Every fancy technique you'll read about later (parent-document retrieval, contextual chunking, RAPTOR, PageIndex) is ultimately a response to one of these two losses. §2 fixes #2 by keeping the document hierarchy. §3 fixes both by pre-synthesizing.


In [ ]:
def chunk_markdown(text: str, max_chars: int = 1200, overlap: int = 150) -> list[str]:
    """Cut a markdown blob into overlapping chunks at paragraph boundaries.

    The job: vector RAG needs *small* pieces of text to embed and retrieve. Too
    big and a chunk is too vague (a 5000-char chunk about ML covers too much
    ground for cosine similarity to be useful). Too small and you lose context
    (a single sentence often makes no sense without its surroundings). 1200
    chars is a common sweet spot for English prose — roughly 200-300 words.

    The algorithm in two sentences: walk paragraph by paragraph, accumulate
    them into a buffer until adding the next paragraph would overshoot
    max_chars; emit the buffer as a chunk; start a new buffer with the LAST
    `overlap` chars of the previous chunk so a sentence that straddles a
    boundary survives in both halves.

    The overlap is the safety net. Without it, "The wiki uses Concept pages.
    Concept pages have YAML frontmatter." could be split right between the two
    sentences, and a query about "concept page frontmatter" would have to
    retrieve BOTH chunks to answer — which top-k often won\'t.

    Args:
        text: The full markdown to chunk.
        max_chars: Soft upper bound per chunk.
        overlap: How many trailing chars to seed the next chunk with.

    Returns:
        List of chunk strings, each at most ~max_chars long. Chunk N+1 starts
        with the last `overlap` chars of chunk N (modulo paragraph boundaries).
    """
    # Split by blank lines first (paragraph boundaries)
    paras = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks, buf = [], ''
    for p in paras:
        if len(buf) + len(p) + 2 > max_chars and buf:
            chunks.append(buf)
            # Keep last `overlap` chars as seed for next chunk
            buf = buf[-overlap:] + '\n\n' + p
        else:
            buf = (buf + '\n\n' + p) if buf else p
    if buf: chunks.append(buf)
    return chunks

chunks = chunk_markdown(markdown_text)
print(f"Produced {len(chunks)} chunks")
print(f"Avg chunk size: {sum(len(c) for c in chunks) // len(chunks)} chars")
print(f"Chunk 0 preview:\n{chunks[0][:240]}...")


### 1.3 Embed + store in ChromaDB

`sentence-transformers/all-MiniLM-L6-v2` runs locally — 384-dimensional vectors, no API calls, ~90 MB model. For a ~100-page single document this is more than enough; you'd reach for a bigger model (`bge-large`, OpenAI `text-embedding-3-small`) when you have millions of chunks and small differences in recall start to matter.

What "384-dim" actually means: every chunk becomes a point in a 384-dimensional space, arranged such that *semantically similar* chunks land near each other. "How do I ingest a document?" lands near a chunk that talks about ingesting, even without keyword overlap. That's the whole bet of vector RAG in one sentence.

ChromaDB's `EphemeralClient` stores everything in RAM — data dies when the kernel dies. Fine for a notebook; swap to `PersistentClient(path=...)` if you want it on disk.


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

chroma = chromadb.EphemeralClient()
collection = chroma.get_or_create_collection(name='article_rag')

# Embed + add
vectors = embedder.encode(chunks, show_progress_bar=False).tolist()
collection.upsert(
    ids=[f"c{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=vectors,
)
print(f"ChromaDB: {collection.count()} chunks stored, 384-dim vectors.")

### 1.4 The query function

Retrieve top-k chunks (k=4) by cosine similarity, concatenate them with `---` separators, ask Claude to answer using **only** that context. Two choices worth understanding:

- **k=4 is a guess.** Lower = less context = less hallucination risk but more misses. Higher = more context = more noise. For a short document, 3–5 is the usual sweet spot.
- **"Use ONLY the context"** is a prompt-engineering guardrail. Without it, Claude happily answers from its general training knowledge, which defeats the point of RAG (you'd have no idea which answers came from the document vs its memory). The downside: if retrieval whiffs, the model says "not covered" instead of trying harder — which for us is a feature, not a bug.


In [ ]:
def answer_vector_rag(question: str, k: int = 4) -> dict:
    """Run the §1 vector-RAG pipeline end-to-end on one question.

    Five steps, each a one-liner:

    1. Embed the question with the same all-MiniLM-L6-v2 model used for the
       chunks. (Critical: query and corpus must be embedded by the SAME model
       — otherwise their vectors live in incomparable spaces.)
    2. Ask ChromaDB for the top-k nearest chunks by cosine similarity.
    3. Glue the retrieved chunks together with `---` separators.
    4. Send to Sonnet with a "use ONLY this context" instruction.
    5. Return the answer plus a trace (which chunks were used) plus the cost
       numbers for the §4 scorecard.

    The whole thing rises and falls on step 2. If cosine similarity returns
    the wrong chunks, even Sonnet can\'t recover. Most "RAG didn\'t work"
    complaints in the wild are actually retrieval failures masquerading as
    generation failures — always look at the trace before blaming the model.

    Args:
        question: Natural-language question.
        k: How many chunks to retrieve (default 4 — see §1.4 markdown for the
           tradeoff).

    Returns:
        Dict with: system, answer, trace (list of chunk previews), latency_s,
        tokens_in, tokens_out. Same schema as the §2 and §3 query functions
        so §4 can compare them apples-to-apples.
    """
    t0 = time.time()
    q_vec = embedder.encode([question]).tolist()
    hits = collection.query(query_embeddings=q_vec, n_results=k)
    retrieved = hits['documents'][0]
    context = '\n\n---\n\n'.join(retrieved)
    prompt = (
        f"Answer the question using ONLY the provided context. If the answer isn\'t in the context, "
        f"say \'Not covered in the retrieved passages.\'\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:"
    )
    msg, _ = ask_claude(prompt, model=MODEL_ANSWER, max_tokens=500)
    return {
        'system': 'vector-rag',
        'answer': msg.content[0].text,
        'trace': [f"chunk {i}: {c[:80]}..." for i, c in enumerate(retrieved)],
        'latency_s': time.time() - t0,
        'tokens_in': msg.usage.input_tokens,
        'tokens_out': msg.usage.output_tokens,
    }

# smoke test
demo = answer_vector_rag(QUESTIONS[0]['question'])
print(f"Q: {QUESTIONS[0]['question']}\n")
print(f"A: {demo['answer']}\n")
print(f"Retrieved: {len(demo['trace'])} chunks  |  "
      f"{demo['latency_s']:.2f}s  |  {demo['tokens_in']+demo['tokens_out']} tokens")


---

## Section 2 — Vectorless RAG (hosted PageIndex + Claude retrieval)

We use [VectifyAI's PageIndex](https://github.com/VectifyAI/PageIndex) directly via their hosted API. The pipeline:

```
PDF → submit_document → poll → get_tree → {structure: [nodes...]}
                                                  ↓
       query → Claude reads tree → picks node_ids → fetches their content → Claude answers
```

PageIndex's pitch in one line: *similarity ≠ relevance*. Cosine similarity finds chunks that *look* alike. Reasoning over a tree finds chunks that *are about* your question. Their hosted service builds the hierarchical index for us once (server-side, ~30–90 seconds for a typical document) and we do every per-query LLM call with Claude — same Haiku/Sonnet tier as everywhere else in this notebook.

We're going to walk through this in small steps so you can *see* what each stage produces. Submit a PDF, watch it process, look at the tree, do one tree search, look at the LLM's reasoning, then put it all together. The point of §2 is to demystify "vectorless RAG" — it's really just a Table of Contents the LLM reasons over.

**One-time setup:** grab a free key from [dash.pageindex.ai/api-keys](https://dash.pageindex.ai/api-keys) and put it in `.env` as `PAGEINDEX_API_KEY=...`.


### 2.1 Initialize the client and upload the PDF

`PageIndexClient` is a thin HTTP wrapper around their hosted API — it doesn't do any LLM work itself. The work happens on their server when you call `submit_document(...)`: they parse the PDF, run their own internal LLM to detect the document's structure, and start building the tree. The call returns immediately with a `doc_id`; the actual processing is async.

Think of `submit_document` as kicking off a background job. We'll poll for completion in the next cell.


In [ ]:
from pageindex import PageIndexClient

assert os.environ.get('PAGEINDEX_API_KEY'), (
    "PAGEINDEX_API_KEY is missing. Free key at https://dash.pageindex.ai/api-keys, then add to .env."
)

pi_client = PageIndexClient(api_key=os.environ['PAGEINDEX_API_KEY'])
print("PageIndex client ready.")

# Kick off the upload + tree-build job. Returns immediately with a doc_id.
print(f"\nUploading {PDF_PATH.name} ({PDF_PATH.stat().st_size / 1e6:.1f} MB)...")
submit_result = pi_client.submit_document(str(PDF_PATH))
doc_id = submit_result['doc_id']
print(f"  doc_id: {doc_id}")
print("  (Save this ID — it's the handle for every subsequent call.)")


### 2.2 Wait for indexing to finish

PageIndex processes the document on their side: PDF → page-aware text extraction → hierarchy detection → per-node summarization. For a short document this takes 30–60 seconds; for a 200-page filing maybe 2–3 minutes.

The polling pattern below is the standard way to wait on async jobs without blocking forever. We check `status` every 5 seconds; the only states we act on are `completed` (we move on) and `failed` (we crash with a clear error so a student knows what went wrong).


In [ ]:
print("Building tree", end='', flush=True)
while True:
    status = pi_client.get_document(doc_id).get('status')
    print('.', end='', flush=True)
    if status == 'completed':
        print(' done')
        break
    if status == 'failed':
        raise RuntimeError(f"PageIndex failed to process {PDF_PATH.name}")
    time.sleep(5)


### 2.3 Pull the tree and look at what we got

`get_tree(doc_id, node_summary=True)` fetches the hierarchical index PageIndex just built. The result is a list of top-level nodes; each node has children (the deeper headings). Two things worth noticing:

- **Stable string IDs** like `"0000"`, `"0001"`. These are document-order indexes — sibling nodes get adjacent IDs, which means the tree is easy to scan visually.
- **Real PDF page numbers** in `start_index` / `end_index`. Citations come back as "pages 21–22" — something a chunk-based system in §1 can't do, because chunks lose page boundaries.

Let's peek at the first node's raw JSON, then walk the whole tree to get a feel for the shape.


In [ ]:
tree_result = pi_client.get_tree(doc_id, node_summary=True)
PAGEINDEX_TREE = tree_result.get('result', [])
print(f"Tree: {len(PAGEINDEX_TREE)} top-level nodes\n")

print("--- first top-level node (raw JSON, children abbreviated) ---")
sample = {k: (f'<{len(v)} children>' if k == 'nodes' else v) for k, v in PAGEINDEX_TREE[0].items()}
print(json.dumps(sample, indent=2))


In [ ]:
def show_tree(tree, depth=0, max_depth=3):
    """Print the tree as an indented outline. Stops descending past max_depth so
    deeply nested documents don't flood the cell output."""
    for node in tree:
        indent = '  ' * depth
        title = node['title'][:70]
        pages = f"p{node.get('start_index', '?')}–{node.get('end_index', '?')}"
        print(f"{indent}[{node['node_id']}] {title}  ({pages})")
        if depth < max_depth and node.get('nodes'):
            show_tree(node['nodes'], depth + 1, max_depth)


def count_nodes(tree):
    return sum(1 + count_nodes(n.get('nodes', [])) for n in tree)


print(f"--- full tree ({count_nodes(PAGEINDEX_TREE)} nodes total) ---\n")
show_tree(PAGEINDEX_TREE)


### 2.4 Tree search — the LLM picks the right nodes

This is where vectorless RAG actually earns its name. We hand the LLM the *whole* tree (titles + summaries, ~2k tokens for a typical document) and ask: "given this question, which node IDs should I read?"

Compare this to §1: there, retrieval is a similarity score in 384-dim space — you can't ask the model "why did you pick chunk 17?" and get a real answer. Here, we explicitly ask the model for its `thinking` *before* it lists IDs. That trace is PageIndex's "no more vibe retrieval" promise: every retrieval is reasoned, not approximated.

We use Haiku for this call. It's a routing decision — cheap, fast, and the model just has to point at the right sections, not synthesize an answer. Sonnet would also work but at ~10x the cost for the same outcome.


In [ ]:
def render_tree_for_llm(tree, depth=0):
    """Recursively render the nested tree as an indented outline for the LLM picker."""
    lines = []
    for node in tree:
        indent = '  ' * depth
        summary = (node.get('summary') or '').strip().replace('\n', ' ')[:200]
        lines.append(f"{indent}[{node['node_id']}] {node['title']} — {summary}")
        if node.get('nodes'):
            lines.append(render_tree_for_llm(node['nodes'], depth + 1))
    return '\n'.join(filter(None, lines))


# Render once. The tree doesn't depend on the question, so the result is reused
# across all 10 §4 showdown queries.
RENDERED_TREE = render_tree_for_llm(PAGEINDEX_TREE)


def llm_tree_search(question):
    """Ask Claude Haiku which tree nodes are likely to answer the question.

    Returns {thinking, node_list, _usage} — PageIndex's explainability shape.
    """
    msg, latency = ask_claude(
        system=(
            "You are navigating a long document via its hierarchical Table of Contents "
            "(produced by PageIndex). Given the tree of sections (with one-line summaries) "
            "and a question, identify the node IDs whose content most likely answers the question. "
            "Think step by step before listing IDs. Return ONLY this JSON:\n"
            '{"thinking": "<step-by-step reasoning>", "node_list": ["0001", "0017"]}'
            "\nUse the exact node IDs as printed in the tree. No prose outside the JSON."
        ),
        prompt=f"DOCUMENT TREE:\n{RENDERED_TREE}\n\nQUESTION: {question}\n\nReturn JSON:",
        model=MODEL_CHEAP, max_tokens=600,
    )
    parsed = parse_json_blob(msg.content[0].text)
    parsed['_usage'] = {
        'tokens_in': msg.usage.input_tokens,
        'tokens_out': msg.usage.output_tokens,
        'latency_s': latency,
    }
    return parsed


In [ ]:
# Standalone test: just the tree-search step, no answer generation.
sample_q = QUESTIONS[0]['question']
print(f"Q: {sample_q}\n")

result = llm_tree_search(sample_q)
print(f"Thinking:\n  {result.get('thinking', '(none)')}\n")
print(f"Picked node IDs: {result.get('node_list', [])}")
print(f"({result['_usage']['in']} in / {result['_usage']['out']} out tokens, "
      f"{result['_usage']['latency_s']:.2f}s)")


### 2.5 Fetch the picked nodes and generate the answer

Two helpers and one orchestrator:

- **`find_nodes(ids)`** — look up the picked node objects by ID. Trivial because we've already flattened the tree into a dict; written out anyway so the pipeline is readable end to end.
- **`generate_answer(question, nodes)`** — concatenate the picked nodes' content with `---` separators, send to Sonnet with the same "use ONLY this context" guardrail as §1 and §3. Citations come back with page numbers because each node carries `start_index` / `end_index`.
- **`answer_vectorless_rag(question, verbose=False)`** — the complete pipeline: tree-search → find → answer. Returns the same dict shape as the other two systems so §4's showdown loop just works.

The `verbose=True` flag prints each step's output (reasoning trace, picked sections, final answer) — useful when you're hand-testing. §4's showdown calls with `verbose=False` so the loop output stays clean.


In [ ]:
NODE_INDEX = {}
def _build_node_index(tree):
    for n in tree:
        NODE_INDEX[n['node_id']] = n
        if n.get('nodes'):
            _build_node_index(n['nodes'])
_build_node_index(PAGEINDEX_TREE)


def find_nodes(node_ids):
    """Look up picked nodes by their IDs. Drops anything not in the tree."""
    return [NODE_INDEX[nid] for nid in node_ids if nid in NODE_INDEX]


def generate_answer(question, nodes):
    """Send picked nodes + question to Sonnet. Returns {text, tokens_in, tokens_out, latency_s}."""
    if not nodes:
        return {'text': 'No relevant sections found in the document.',
                'tokens_in': 0, 'tokens_out': 0, 'latency_s': 0.0}

    chunks = []
    for n in nodes:
        pages = f"pages {n.get('start_index', '?')}–{n.get('end_index', '?')}"
        body = n.get('text') or n.get('summary', '')
        chunks.append(f"# {n['title']} ({pages})\n{body}")
    context = '\n\n---\n\n'.join(chunks)

    msg, latency = ask_claude(
        f"Answer using ONLY the provided context. For every claim, cite the section "
        f"title and page range in parentheses. If the answer isn't in the context, "
        f"say 'Not covered in the selected sections.'\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:",
        model=MODEL_ANSWER, max_tokens=500,
    )
    return {
        'text': msg.content[0].text,
        'tokens_in': msg.usage.input_tokens,
        'tokens_out': msg.usage.output_tokens,
        'latency_s': latency,
    }


def answer_vectorless_rag(question, verbose=False):
    """The complete §2 pipeline: tree-search → find nodes → generate answer.

    Mirrors Krishnaik's `vectorless_rag` flow with Claude end-to-end.
    Set verbose=True for hand-testing; §4's showdown loop calls with verbose=False.
    """
    t0 = time.time()
    if verbose:
        print(f"{'=' * 55}\nQ: {question}\n{'=' * 55}")

    search = llm_tree_search(question)
    node_ids = [str(nid) for nid in search.get('node_list', []) if str(nid) in NODE_INDEX][:8]
    thinking = search.get('thinking', '(no reasoning emitted)')
    if verbose:
        print(f"\nReasoning:\n  {thinking[:400]}{'...' if len(thinking) > 400 else ''}")
        print(f"\nPicked nodes: {node_ids}")

    nodes = find_nodes(node_ids)
    if verbose:
        for n in nodes:
            print(f"  - [{n['node_id']}] {n['title']} (pages "
                  f"{n.get('start_index', '?')}–{n.get('end_index', '?')})")

    ans = generate_answer(question, nodes)
    if verbose:
        print(f"\nAnswer:\n{ans['text']}")

    return {
        'system': 'vectorless-rag',
        'answer': ans['text'],
        'reasoning': thinking,
        'trace': [f"node {n['node_id']}: {n['title']}" for n in nodes],
        'latency_s': time.time() - t0,
        'tokens_in': search['_usage']['tokens_in'] + ans['tokens_in'],
        'tokens_out': search['_usage']['tokens_out'] + ans['tokens_out'],
    }


In [ ]:
# Smoke test: full pipeline on the first question, verbose so you can see every step.
demo2 = answer_vectorless_rag(QUESTIONS[0]['question'], verbose=True)
print(f"\n{'=' * 55}")
print(f"Total: {demo2['latency_s']:.2f}s  |  "
      f"{demo2['tokens_in'] + demo2['tokens_out']:,} tokens "
      f"({demo2['tokens_in']:,} in / {demo2['tokens_out']:,} out)")


---

## Section 3 — Self-Writing Wiki (Karpathy Pattern, From Scratch)

```
PDF  →  Claude reads CLAUDE.md (schema)  →  runs 9-step Ingest workflow
                                            writes Sources/, Wiki/, System/ pages
                                                         ↓
                       query → Claude reads System/Index.md → opens 1-3 Wiki pages → answers
                              (NEVER re-reads Raw during Query. If the wiki doesn't know, it says so.)
```

The thesis: classic RAG re-derives understanding every query. The wiki pays that cost once, at ingest time, and the understanding *persists as markdown*. The Lint and Prune operations keep it fresh when new sources contradict old ones.

**We drive this with the vault's own rulebook** — `Knowledge-Base/CLAUDE.md`. It defines the folder layout (Raw/ Sources/ Wiki/ System/ Archive/), the page templates (Concept / Product / Persona / Analysis), the 9-step Ingest workflow, the 5-step Query workflow, the Lint + Prune rules, and the things Claude must never do. Section 3 loads that file verbatim as the system prompt — nothing is invented here.


### 3.1 Seed the System/ dashboards (Raw already holds the source from §1.1)

Five folders ship with the repo: **Raw/**, **Sources/**, **Wiki/**, **System/**, **Archive/**. The Docling-parsed source landed in `Raw/article.md` back in §1.1 — that's the "ingest-to-Raw" step from `CLAUDE.md` §0, and it's what the wiki is about to ingest.

What's left to do before the 9-step ingest fires: reset the `System/` dashboards (Index, Glossary, Overview, Log) to a clean slate so each notebook run starts reproducibly. That's this cell.


In [ ]:
# VAULT was pinned in cell 5. §1.1 already parsed article.pdf → Raw/article.md via Docling
# (the "ingest-to-Raw" step from CLAUDE.md §0). Here we just:
#   (a) assert Raw/article.md exists so a student running §3 alone gets a helpful error;
#   (b) re-seed the System/ dashboards so each run starts from a clean state.

raw_source = VAULT / 'Raw' / 'article.md'
assert raw_source.exists(), (
    f"Missing {raw_source.relative_to(HERE)}. Run §1.1 first — Docling needs to "
    f"parse article.pdf and drop the markdown into Raw/."
)

(VAULT / 'System' / 'Index.md').write_text(
    "# Index\n\n> Master catalog of all pages. Updated on every ingest.\n\n"
    "## Sources\n\n_none yet_\n\n## Concepts\n\n_none yet_\n\n"
    "## Products\n\n_none yet_\n\n## Personas\n\n_none yet_\n\n"
    "## Analysis\n\n_none yet_\n\n## Archive\n\n_none yet_\n",
    encoding='utf-8')
(VAULT / 'System' / 'Glossary.md').write_text(
    "# Glossary\n\n> Terms and aliases used across the vault. Alphabetical.\n",
    encoding='utf-8')
(VAULT / 'System' / 'Overview.md').write_text(
    "# Overview\n\n> Big-picture synthesis. Updated only when the theme shifts.\n",
    encoding='utf-8')
(VAULT / 'System' / 'Log.md').write_text(
    "# Log\n\n> Append-only record of every ingest / query / lint / prune. "
    "ISO timestamps. Most recent at the bottom.\n",
    encoding='utf-8')

print(f"Vault at: {VAULT.relative_to(HERE)}/")
for p in sorted(VAULT.rglob('*')):
    if p.is_dir() or p.name == '.gitkeep':
        continue
    rel = p.relative_to(VAULT)
    print(f"  {rel}  [{p.stat().st_size:,} bytes]")


### 3.2 Ingest — Claude reads `CLAUDE.md`, then runs the 9-step workflow

This is the key difference from a generic RAG ingest. We pass the vault's **actual schema** (`Knowledge-Base/CLAUDE.md`) as the system prompt, verbatim. Claude already knows the folder layout, the naming rules, the page templates, the 9 ingest steps, and the 10 things it must never do — because we just dropped the rulebook in its lap.

We add a short **transport note**: this session is non-interactive; output the whole vault state as a single JSON object keyed by filepath. That detail matters. Without it, Claude would try to run the interactive flow from `CLAUDE.md` §4 (pause for "anything I should emphasize?", confirm between steps) — which no notebook cell can honor. Trading one big JSON blob for the usual tool-use loop is the trick that lets us reuse the exact schema that drives Rahul's real Cowork sessions.


In [ ]:
# Read the actual rulebook — the same one living at VAULT/CLAUDE.md
schema_text = (VAULT / 'CLAUDE.md').read_text(encoding='utf-8')

INGEST_TRANSPORT_NOTE = '''

---

## Transport note (this session only)

You're running non-interactively — there is no human in the loop between steps. So:

- Do NOT pause between steps 5 and 6 for confirmation (CLAUDE.md §4). Run all 9 steps in one shot.
- Do NOT ask "anything I should emphasize?" — skip the discussion in step 3.
- Instead of writing files directly, output a SINGLE JSON object where each KEY is a relative path inside the vault (e.g. "Sources/2023-11-01 — Data-Driven Organization.md", "Wiki/Concept — Data Mesh.md", "System/Index.md") and each VALUE is the full markdown body for that file.
- Include updated versions of System/Index.md, System/Glossary.md, System/Overview.md, and an appended entry to System/Log.md (return the FULL new Log.md contents, not a diff).
- Follow §3 page templates exactly (YAML frontmatter + required sections).
- Follow §8 naming rules exactly (em-dash "—" not hyphen). Use the singular role-based form for personas (e.g., "Persona — MLOps Practitioner", not "Persona — Data Platform Team"). Do not put parenthetical aliases inside Concept/Product titles — put aliases in the frontmatter `aliases:` field instead.
- Em-dashes in filenames and wikilinks: use the actual U+2014 character.
- Output ONLY the JSON object — no prose before or after, no markdown fences.
'''

INGEST_SYSTEM = schema_text + INGEST_TRANSPORT_NOTE

def ingest_pdf_as_wiki() -> dict:
    """Run the 9-step Ingest workflow from CLAUDE.md §4 — in a single LLM call.

    The wiki\'s killer move and the most expensive call in the notebook.

    What\'s happening conceptually:

    We hand Sonnet two things in one prompt:

    - The vault\'s own rulebook (CLAUDE.md, ~3k tokens) as the SYSTEM message.
      This tells Sonnet exactly how the vault is structured, what page
      templates to follow, what naming rules to use, what NOT to do.
    - The Docling-parsed source markdown as the USER message.

    We then ask: "Run all 9 ingest steps and emit the resulting vault state
    as JSON keyed by relative file path." Sonnet does step 1 (read source),
    step 4 (write Source Summary), step 5 (create/update Concept/Product/
    Persona pages), and steps 6-9 (update Index/Glossary/Overview/Log) —
    outputting the whole thing as a single JSON object.

    **Why JSON-as-transport instead of agentic file writes?** CLAUDE.md §4
    has Sonnet pause for human input between steps ("anything I should
    emphasize?", confirm between Index/Overview/Log writes). A notebook can\'t
    honor those pauses, so we collapse the multi-step interactive flow into a
    single non-interactive emit. The transport note in INGEST_SYSTEM tells
    Sonnet to skip the human-in-the-loop bits for THIS session only — the
    rulebook itself stays untouched, so it remains exactly the schema that
    drives Rahul\'s real Cowork sessions.

    **Why one big call?** Two reasons. First, every wiki page needs to be
    consistent with the others (same Concept page should be linked the same
    way from the Source Summary, the Index, and the Glossary). Doing this in
    N independent calls would risk drift. Second, prompt caching
    (`cache_system=True`) makes the 3k-token CLAUDE.md system prompt cheap on
    reruns — a one-shot is the natural shape.

    **Cost.** ~30k input tokens + up to 16k output tokens, ~3-5 minutes
    wall-clock on first call. Reruns within 5 minutes get the system prompt
    served from cache (90% discount on those tokens, faster TTFT).

    **Failure modes.**
    - Sonnet may emit truncated JSON if it bumps the 16k output ceiling
      (rare for one document; happens when the source is very long and
      detailed). The parser tolerates stray markdown fences but not
      mid-string truncation.
    - Em-dashes in filenames may get normalized to hyphens by some
      intermediaries — the writer code preserves whatever Sonnet returns
      and §3.4 has fuzzy filename matching to recover.
    - We forbid the model from writing CLAUDE.md or anything in Raw/ (per
      the rulebook §10); attempts get logged and skipped.

    Returns:
        Dict with: pages_written (list of relative paths), latency_s,
        tokens_in, tokens_out, cache_read, cache_create.
    """
    t0 = time.time()
    # Defensive re-resolve so cell 30 works even if cell 28 hasn't run this kernel.
    src_path = VAULT / 'Raw' / 'article.md'
    source_text = src_path.read_text(encoding='utf-8')
    source_rel = src_path.relative_to(VAULT).as_posix()
    if len(source_text) > 120_000:
        print(f"  warn: source is {len(source_text):,} chars; truncating to 120k for the ingest call")

    print("  streaming ingest output:", end='', flush=True)
    msg, _ = ask_claude(
        system=INGEST_SYSTEM,
        prompt=(
            f"INGEST TRIGGER: ingest {source_rel}\n\n"
            f"RAW SOURCE CONTENT (this is the file at {source_rel}):\n\n"
            f"{source_text[:120_000]}\n\n"
            f"Now run the 9-step Ingest workflow (CLAUDE.md §4) and output the JSON."
        ),
        model=MODEL_ANSWER, max_tokens=16000,
        stream=True,         # dots as tokens arrive
        cache_system=True,   # cache the ~3k-token CLAUDE.md schema for 5-min reruns
    )
    u = msg.usage
    # Parse JSON, tolerant of stray fences
    text = msg.content[0].text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.MULTILINE).strip()
    pages = parse_json_blob(text)
    if not pages:
        raise RuntimeError(f"No JSON in response:\n{text[:500]}")

    # Write every page to disk, but NEVER overwrite CLAUDE.md or anything under Raw/
    written = []
    for rel_path, content in pages.items():
        rel_path = rel_path.lstrip('/').replace('\\', '/')
        if rel_path == 'CLAUDE.md' or rel_path.startswith('Raw/'):
            print(f"  SKIP (protected): {rel_path}")
            continue
        target = VAULT / rel_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding='utf-8')
        written.append(rel_path)

    return {
        'pages_written': written,
        'latency_s': time.time() - t0,
        'tokens_in': u.input_tokens,
        'tokens_out': u.output_tokens,
        'cache_read': u.cache_read_input_tokens or 0,
        'cache_create': u.cache_creation_input_tokens or 0,
    }

ingest_result = ingest_pdf_as_wiki()
cache_bit = ""
if ingest_result.get('cache_read'):
    cache_bit = f", cache hit {ingest_result['cache_read']:,} tok"
elif ingest_result.get('cache_create'):
    cache_bit = f", cached {ingest_result['cache_create']:,} tok for next run"
print(f"\nIngest done in {ingest_result['latency_s']:.1f}s  "
      f"({ingest_result['tokens_in']:,} in / {ingest_result['tokens_out']:,} out tokens{cache_bit})")
print(f"\nWrote {len(ingest_result['pages_written'])} pages:")
for p in sorted(ingest_result['pages_written']): print(f"  - {p}")

### 3.3 Peek at what the LLM wrote

Sanity-check: did it follow the schema? We print `System/Index.md` (should be grouped by type, should list the real concept/product/persona pages) and `System/Log.md` (should have a single ingest entry appended).

In [ ]:
for rel in ['System/Index.md', 'System/Log.md']:
    p = VAULT / rel
    if p.exists():
        print(f"============ {rel} ============")
        print(p.read_text(encoding='utf-8'))
        print()

### 3.4 The query function — `CLAUDE.md` §5 Query workflow

From the rulebook (§5):

1. Read `System/Index.md` first.
2. Identify 1–3 candidate Wiki pages.
3. Read those pages (full, not grep).
4. Compose the answer. Cite with `[[Page Name]]` inline.
5. "Do NOT re-read Raw during Query. The whole point is that Wiki is the distillation — if the wiki doesn't know, the gap is what matters and the right answer is 'I don't know, consider ingesting a source on X.'"

That's what we implement here. Two LLM calls: pick pages, then answer.

In [ ]:
# Read Index + Glossary once — they're generated by the §3.2 ingest and don't change per query.
WIKI_INDEX = (VAULT / 'System' / 'Index.md').read_text(encoding='utf-8')
WIKI_GLOSSARY = (VAULT / 'System' / 'Glossary.md').read_text(encoding='utf-8')


def answer_wiki(question):
    """Run the §5 Query workflow from CLAUDE.md — pick wiki pages, then answer.

    Two LLM calls:
    1. PICK (Haiku) — read Index + Glossary as the system prompt (cached, so the ~7k tokens are
       sent once and reused across all 10 §4 questions). Ask for 1–3 wiki page filenames as JSON.
    2. ANSWER (Sonnet) — read picked pages in full (per CLAUDE.md §5: "full, not grep"). Cite
       inline with [[Page Name]] wikilinks. Refuse cleanly if the wiki doesn't cover the topic.

    The wiki has been pre-synthesized at ingest, so synthesis questions tend to win here.
    Raw/ is intentionally off-limits (§5 forbids it) — missing coverage surfaces as a refusal.
    """
    t0 = time.time()

    pick, _ = ask_claude(
        system=(
            "You follow CLAUDE.md §5 Query workflow. Below are the Index and Glossary of a "
            "markdown wiki. When the user asks a question, identify 1–3 Wiki page filenames "
            "(relative paths, e.g. 'Wiki/Concept — RAG.md') most likely to contain the answer. "
            "Return ONLY a JSON array of filenames. No prose.\n\n"
            f"INDEX:\n{WIKI_INDEX}\n\nGLOSSARY:\n{WIKI_GLOSSARY}"
        ),
        prompt=f"QUESTION: {question}\n\nReturn JSON array of page filenames:",
        model=MODEL_CHEAP, max_tokens=300,
        cache_system=True,   # Index + Glossary are static across all queries — 90% discount
    )
    files = parse_json_blob(pick.content[0].text, array=True)

    # Read picked pages full (never grep, never re-read Raw)
    loaded, missing = [], []
    for f in files[:3]:
        rel = f.lstrip('/').replace('\\', '/')
        if rel.startswith('Raw/'):      # §5 forbids re-reading Raw
            missing.append(f"FORBIDDEN_RAW:{rel}"); continue
        candidate = VAULT / rel
        if candidate.exists() and candidate.is_file():
            loaded.append((str(candidate.relative_to(VAULT)),
                           candidate.read_text(encoding='utf-8')))
        else:
            missing.append(f)

    context = '\n\n---\n\n'.join(f"FILE: {name}\n{body}" for name, body in loaded) or WIKI_INDEX

    msg, _ = ask_claude(
        system=(
            "You follow CLAUDE.md §5 Query workflow. Answer using ONLY the wiki content "
            "provided. Cite pages inline with [[Page Name]] wikilinks. "
            "If the wiki does not cover the topic, reply exactly: "
            "\"I don't know — consider ingesting a source on this topic.\" "
            "Do NOT invent facts not in the provided pages."
        ),
        prompt=f"WIKI CONTENT:\n{context}\n\nQUESTION: {question}\n\nANSWER:",
        model=MODEL_ANSWER, max_tokens=500,
    )
    return {
        'system': 'wiki',
        'answer': msg.content[0].text,
        'trace': [name for name, _ in loaded] + [f"MISSING:{x}" for x in missing],
        'latency_s': time.time() - t0,
        'tokens_in': pick.usage.input_tokens + msg.usage.input_tokens,
        'tokens_out': pick.usage.output_tokens + msg.usage.output_tokens,
    }


demo3 = answer_wiki(QUESTIONS[0]['question'])
print(f"Q: {QUESTIONS[0]['question']}\n")
print(f"A: {demo3['answer']}\n")
print(f"Pages opened: {demo3['trace']}")
print(f"{demo3['latency_s']:.2f}s  |  {demo3['tokens_in']+demo3['tokens_out']} tokens")


---

## Section 4 — The 10-Question Showdown

All three systems, all 10 questions, scored by Claude Opus. We measure:

- **Accuracy** — does the answer match the expected keywords / meaning? (0-10)
- **Faithfulness** — is the answer grounded, or did it hallucinate? (0-10)
- **Refusal calibration** — for out-of-scope, did it correctly say "not covered"? (0-10)
- **Latency** (seconds) and **tokens** (cost proxy)

Plus per-question retrieval trace so you can see *why* a system failed.

### 4.1 Run all systems × all questions

This is where your Anthropic bill jumps. ~3 LLM calls per row × 30 rows ≈ 90 calls. Mostly haiku.

In [ ]:
SYSTEMS = [
    ('vector-rag',      answer_vector_rag),
    ('vectorless-rag',  answer_vectorless_rag),
    ('wiki',            answer_wiki),
]

rows = []
for q in QUESTIONS:
    for sys_name, fn in SYSTEMS:
        print(f"  {sys_name:<16s} ← {q['id']}")
        try:
            r = fn(q['question'])
        except Exception as e:
            r = {'system': sys_name, 'answer': f'ERROR: {e}',
                 'trace': [], 'latency_s': 0, 'tokens_in': 0, 'tokens_out': 0}
        rows.append({**q, **r})

print(f"\nCollected {len(rows)} (question, system) results.")

### 4.2 Judge with Opus

For each answer we give Opus: the question, the expected keywords, the system's answer, and a rubric (accuracy / faithfulness / refusal calibration). It returns JSON scores.

LLM-as-judge is imperfect. Three things to watch for when you read §4.5:

- **Bias toward wordy answers.** Judges often score confident prose higher than a correct-but-terse answer. Cross-check the short ones manually.
- **Poor at detecting hallucination.** A judge that didn't see the source document scores "faithfulness" by feel, not by checking citations. If you want bulletproof faithfulness, have the judge re-read the source — this notebook doesn't, to save cost.
- **Self-preference.** An Opus judge over Sonnet answers is less biased than the reverse would be, but not zero-biased. The mitigation is to use a judge meaningfully stronger than any of the generators, which we do.


In [ ]:
JUDGE_SYSTEM = '''You are a strict evaluator of RAG systems.
Given a question, expected keywords/facts, and a system's answer, score it.

Return JSON only:
{
  "accuracy":   <0-10>,  // does the answer contain the expected facts?
  "faithfulness": <0-10>, // is it grounded or hallucinated?
  "refusal_calibration": <0-10>, // if the question was out-of-scope, did it correctly refuse? Else give 10.
  "notes": "<one sentence>"
}'''

def score_answer(row):
    """Have Opus score one (question, system answer) pair. Returns the JSON dict from the judge.

    Falls back to all-zeros + a parse-error note if the judge's output isn't valid JSON,
    so a single bad output doesn't crash the whole §4 run.
    """
    prompt = (
        f"QUESTION ({row['type']}):\n{row['question']}\n\n"
        f"EXPECTED KEYWORDS / FACTS:\n{row['expected_keywords']}\n\n"
        f"SYSTEM ANSWER:\n{row['answer']}\n\n"
        f"Return your JSON evaluation."
    )
    msg, _ = ask_claude(prompt, system=JUDGE_SYSTEM, model=MODEL_JUDGE, max_tokens=300)
    parsed = parse_json_blob(msg.content[0].text)
    if not parsed:
        return {'accuracy': 0, 'faithfulness': 0, 'refusal_calibration': 0,
                'notes': f'PARSE_ERROR: {msg.content[0].text[:100]}'}
    return parsed


scored_rows = []
for i, row in enumerate(rows):
    print(f"  judging {i+1}/{len(rows)}: {row['id']} / {row['system']}")
    s = score_answer(row)
    scored_rows.append({**row, **s})
print("Done scoring.")


### 4.3 Scorecard

In [ ]:
import pandas as pd

df = pd.DataFrame(scored_rows)
df['total_tokens'] = df['tokens_in'] + df['tokens_out']

# per-system aggregate
agg = df.groupby('system').agg(
    accuracy=('accuracy', 'mean'),
    faithfulness=('faithfulness', 'mean'),
    refusal_calibration=('refusal_calibration', 'mean'),
    avg_latency_s=('latency_s', 'mean'),
    avg_tokens=('total_tokens', 'mean'),
).round(2)
print("\n=== AGGREGATE SCORECARD ===\n")
print(agg)
print("\n=== PER-QUESTION ACCURACY ===\n")
piv = df.pivot(index='id', columns='system', values='accuracy')
print(piv)

### 4.4 Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar: accuracy per question per system
piv.plot.bar(ax=axes[0], width=0.8)
axes[0].set_title('Accuracy by Question (0-10)')
axes[0].set_xlabel('Question')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 10.5)
axes[0].legend(title='System', loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# Aggregate scorecard
agg_plot = agg[['accuracy', 'faithfulness', 'refusal_calibration']]
agg_plot.plot.bar(ax=axes[1])
axes[1].set_title('Aggregate Scores by System')
axes[1].set_ylabel('Score (0-10)')
axes[1].set_ylim(0, 10.5)
axes[1].legend(loc='lower right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 4.5 Read the failures — this is where it gets interesting

Sort by lowest accuracy and look at what each system got wrong. The *pattern* of failures reveals the architecture.

In [ ]:
failures = df[df['accuracy'] < 7].sort_values('accuracy')
for _, row in failures.iterrows():
    print(f"[{row['id']}/{row['type']}] {row['system']}  →  acc={row['accuracy']}, faith={row['faithfulness']}")
    print(f"  Q: {row['question']}")
    print(f"  Expected: {row['expected_keywords']}")
    print(f"  Got: {row['answer'][:180]}...")
    print(f"  Trace: {row['trace'][:3]}")
    print(f"  Judge note: {row['notes']}")
    print()